# AStrategyActorCriticJ

> JAX CRLD actor-critic agents in strategy space. Inherits from strategybaseJ.

In [ ]:
#| default_exp Agents/StrategyActorCriticJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import jax
from jax import jit
import jax.numpy as jnp
import itertools as it
from functools import partial
from fastcore.utils import *

from pyCRLD.Agents.StrategyBaseJ import strategybaseJ
from pyCRLD.Utils.HelpersJ import *

In [ ]:
#| export
class stratACJ(strategybaseJ):
    """Class for JAX CRLD actor-critic agents in strategy space."""

    @partial(jit, static_argnums=(0, 2))
    def RPEisa(self,
               Xisa,       # Joint strategy
               norm=False  # normalize error around actions?
               ) -> jnp.ndarray:  # RP/TD error
        """Reward-prediction error for actor-critic dynamics, given joint strategy `Xisa`."""
        R = self.Risa(Xisa)
        Vis = self.Vis(Xisa, Risa=R)
        NextV = self.NextVisa(Xisa, Vis=Vis)

        n = jnp.newaxis
        E = self.pre[:, n, n] * R + self.gamma[:, n, n] * NextV - Vis[:, :, n]
        E *= self.beta[:, n, n]

        E = E - E.mean(axis=2, keepdims=True) if norm else E
        return E

    @partial(jit, static_argnums=0)
    def NextVisa(self,
                 Xisa,
                 Vis=None,
                 Tss=None,
                 Ris=None,
                 Risa=None) -> jnp.ndarray:
        """Strategy-average next value for agent i, state s, action a."""
        Vis = self.Vis(Xisa, Ris=Ris, Tss=Tss, Risa=Risa) if Vis is None else Vis

        i = 0; a = 1; s = 2; s_ = 3
        j2k = list(range(6, 6 + self.N - 1))
        b2d = list(range(6 + self.N - 1, 6 + self.N - 1 + self.N))
        e2f = list(range(5 + 2 * self.N, 5 + 2 * self.N + self.N - 1))

        sumsis = [[j2k[l], s, e2f[l]] for l in range(self.N - 1)]
        otherX = list(it.chain(*zip((self.N - 1) * [Xisa], sumsis)))

        args = [self.Omega, [i] + j2k + [a] + b2d + e2f] + otherX + \
               [self.T, [s] + b2d + [s_], Vis, [i, s_], [i, s, a]]
        return jnp.einsum(*args, optimize=self.opti)

## Tests

In [ ]:
from pyCRLD.Environments.SocialDilemmaJ import SocialDilemmaJ

env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)
agent = stratACJ(env, learning_rates=0.1, discount_factors=0.9, choice_intensities=1.0)

X = agent.zero_intelligence_strategy()
assert X.shape == (2, 1, 2)

# Test RPEisa
rpe = agent.RPEisa(X)
assert rpe.shape == (2, 1, 2)  # N, Z, M

In [ ]:
# Test trajectory
key = jax.random.PRNGKey(42)
X0 = agent.random_softmax_strategy(key=key)
traj, fpr = agent.trajectory(X0, Tmax=50)
assert traj.shape == (50, 2, 1, 2)  # Tmax, N, Z, M
assert jnp.allclose(traj.sum(-1), 1.0, atol=1e-5)  # valid probabilities

In [ ]:
#|eval: false
import time
_ = agent.trajectory(X0, Tmax=100)
t0 = time.time()
for _ in range(10):
    traj2, _ = agent.trajectory(X0, Tmax=200)
    traj2.block_until_ready()
t1 = time.time()
print(f'10x Tmax=200: {(t1-t0)*1000:.1f}ms total')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()